# Time Petri nets

Transitions carry a firing interval `(eft, lft)`. With integer-time semantics, a transition can fire once it has been continuously enabled for `c` time units with `eft <= c <= lft`, and time may not advance past `lft` while it is enabled. `explore_timed` builds the state space of (marking, clocks), with `"tick"` edges for time steps.

```mermaid
flowchart LR
    Req((Req 1)) --> serve["serve [1,2]"] --> Served((Served))
    Req --> timeout["timeout [3,4]"] --> TimedOut((TimedOut))
```

In [1]:
from petrilab.petri.model import PTNet
from petrilab.petri.timed import explore_timed


def request_net(serve, timeout):
    n = PTNet("request")
    n.add_place("Req", 1)
    n.add_place("Served")
    n.add_place("TimedOut")
    n.add_transition("serve", {"Req": 1}, {"Served": 1}, interval=serve)
    n.add_transition("timeout", {"Req": 1}, {"TimedOut": 1}, interval=timeout)
    return n


n = request_net((1, 2), (3, 4))
print(sorted({t for s in n.reachable().values() for t, _ in s}))

['serve', 'timeout']


Ignoring time, both outcomes are reachable: the untimed net is non-deterministic.

In [2]:
space = explore_timed(n)
print("fired:", space.fired)
print("TimedOut reachable:", n.marking(TimedOut=1) in space.markings)
print("Served reachable:", n.marking(Served=1) in space.markings)

fired: {'serve'}
TimedOut reachable: False
Served reachable: True


With intervals, `serve` becomes urgent at time 2 and time cannot advance to 3, so `timeout` never fires: the timing makes the timeout unreachable.

In [3]:
n = request_net((3, 5), (3, 4))
print(explore_timed(n).fired)

{'serve', 'timeout'}


When the intervals overlap (both can fire at time 3 or 4), both transitions fire in some run.

In [4]:
n = request_net((2, 2), (5, 5))
space = explore_timed(n)
start = next(
    s for s in space.states if s[0] == n.initial_marking and dict(s[1])["serve"] == 0
)
print([label for label, _ in space.edges[start]])

['tick']


At clock 0 the only possible step is a `tick`: `serve` cannot fire before its earliest firing time 2.

In [5]:
n = PTNet("persist")
for p, k in [("A", 1), ("B", 1), ("A2", 0), ("B2", 0)]:
    n.add_place(p, k)
n.add_transition("a", {"A": 1}, {"A2": 1}, interval=(1, 1))
n.add_transition("b", {"B": 1}, {"B2": 1}, interval=(2, 2))
print(n.marking(A2=1, B2=1) in explore_timed(n).markings)

True


Clocks persist across unrelated firings: `b` keeps counting while `a` fires at time 1, then fires at time 2, so both final tokens are reached.